## Prompt Engineering en LangSmith

### Importamos variables de entorno

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path=".env", override=True)
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Ejecutamos un Pull del Prompt desde Prompthub

In [2]:
from langsmith import Client
client = Client(api_key=LANGSMITH_API_KEY)
prompt = client.pull_prompt("eli5-conciso-nuevo", include_model=True)

### Setup Aplicación IA 

Primero configuremos nuestra herramienta de búsqueda web, como siempre.

In [3]:
from langchain_community.tools.tavily_search import TavilySearchResults

web_search_tool = TavilySearchResults(max_results=1,
    tavily_api_key=TAVILY_API_KEY)


C:\Users\sergi\AppData\Local\Temp\ipykernel_15132\1216388379.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults
C:\Users\sergi\AppData\Local\Temp\ipykernel_15132\1216388379.py:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search_tool = TavilySearchResults(max_results=1,


Ahora creemos nuestra aplicación, igual que en el módulo de trazas. Esta vez, nuestro prompt es el que se obtuvo desde PromptHub.


In [4]:
from openai import OpenAI
from langsmith import traceable
from langsmith.wrappers import wrap_openai

# Creamos la aplicación
openai_client = wrap_openai(OpenAI())

@traceable
def search(question):
    web_docs = web_search_tool.invoke({"query": question})
    web_results = "\n".join([d["content"] for d in web_docs])
    return web_results
    
@traceable
async def explain(question, context):
    prompt_inputs = {"question": question, "context": context}
    ai_message = await prompt.ainvoke(prompt_inputs)
    return ai_message.content

@traceable
async def eli5(question):
    context = search(question)
    answer = await explain(question, context)
    return answer

### Test Application

In [5]:
question = "Qué es la biotecnología?"
print(await eli5(question))

La biotecnología es como usar la magia de la naturaleza para hacer cosas útiles. Utilizamos pequeños seres vivos, como bacterias o plantas, para ayudar a crear alimentos, medicinas o productos nuevos. Es como cuando alguien hornea pan, usando ingredientes sencillos, pero con técnicas especiales para hacerlo delicioso. Así, la biotecnología nos ayuda a mejorar nuestra vida y cuidar el planeta.
